In [ ]:
-------------------------tushare----pre------非常规的数据采集和预处理做法----从特征选择开始-------------------

In [3]:
import tushare as ts
import pandas as pd
import numpy as np
import time

TOKEN = "YOUR_TUSHARE_TOKEN"
ts.set_token(TOKEN)
pro = ts.pro_api()

INDEX_LIST = {
    "CSI300": "000300.SH",
    "CSI1000": "000852.SH",
}

START_DATE = "20000101"
END_DATE = "20251114"

def fetch_index_daily(ts_code, start_date, end_date):
    df = pro.index_daily(
        ts_code=ts_code,
        start_date=start_date,
        end_date=end_date,
        fields="ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount"
    )

    print(ts_code, "raw shape:", df.shape)
    print("columns:", df.columns.tolist())

    if df.empty:
        raise ValueError(f"{ts_code} 返回空数据，请检查权限、代码或日期范围")

    if "trade_date" not in df.columns:
        raise ValueError(f"{ts_code} 返回字段异常：{df.columns.tolist()}")

    df = df.sort_values("trade_date").reset_index(drop=True)
    return df

def add_basic_features(df):
    df = df.copy()
    df["trade_date"] = pd.to_datetime(df["trade_date"])

    df["return"] = df["close"].pct_change()
    df["high_low_range"] = (df["high"] - df["low"]) / df["close"]
    df["close_open_range"] = (df["close"] - df["open"]) / df["open"]

    df["volume_ratio"] = df["vol"] / df["vol"].rolling(20).mean()
    df["ma5"] = df["close"].rolling(5).mean()
    df["ma20"] = df["close"].rolling(20).mean()

    df["close_to_ma5"] = df["close"] / df["ma5"] - 1
    df["close_to_ma20"] = df["close"] / df["ma20"] - 1

    df["roc_5"] = df["close"].pct_change(5)
    df["roc_10"] = df["close"].pct_change(10)
    df["roc_20"] = df["close"].pct_change(20)

    df["bb_width"] = df["close"].rolling(20).std() / df["ma20"]

    df["dayofyear"] = df["trade_date"].dt.dayofyear
    df["dayofyear_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365)

    return df.dropna().reset_index(drop=True)

for name, code in INDEX_LIST.items():
    print(f"\nDownloading {name}: {code}")

    df = fetch_index_daily(code, START_DATE, END_DATE)
    df = add_basic_features(df)

    out = f"{name}_index_2000_2025.csv"
    df.to_csv(out, index=False, encoding="utf-8-sig")

    print(f"Saved: {out}")
    print(df.shape)
    print(df["trade_date"].min(), df["trade_date"].max())

    time.sleep(0.5)


000300.SH raw shape: (5068, 11)
columns: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
Saved: CSI300_index_2000_2025.csv
(5048, 25)
2005-02-01 00:00:00 2025-11-14 00:00:00

000852.SH raw shape: (5068, 11)
columns: ['ts_code', 'trade_date', 'open', 'high', 'low', 'close', 'pre_close', 'change', 'pct_chg', 'vol', 'amount']
Saved: CSI1000_index_2000_2025.csv
(5048, 25)
2005-02-01 00:00:00 2025-11-14 00:00:00


In [4]:
"""
股票数据获取脚本 v2 - 使用akshare（稳定，无限速问题）
支持美股、港股
输出格式与A股数据对齐

安装依赖：pip install akshare pandas
"""

import akshare as ak
import pandas as pd
import os
import time

# ============================================================
# 股票池配置
# ============================================================

# 美股医疗（akshare用小写ticker）
US_MEDICAL = {
    "JNJ":  "强生",
    "PFE":  "辉瑞",
    "UNH":  "联合健康",
    "ABT":  "雅培",
    "MRK":  "默沙东",
    "TMO":  "赛默飞",
    "DHR":  "丹纳赫",
    "MDT":  "美敦力",
    "BMY":  "百时美施贵宝",
    "AMGN": "安进",
}

# 美股电力/公用事业
US_POWER = {
    "NEE":  "NextEra能源",
    "DUK":  "杜克能源",
    "SO":   "南方公司",
    "D":    "优势能源",
    "AEP":  "美国电力",
    "EXC":  "Exelon",
    "SRE":  "Sempra",
    "XEL":  "Xcel能源",
    "ES":   "Eversource",
    "WEC":  "WEC能源",
}

# 港股医疗（symbol用5位数字字符串）
HK_MEDICAL = {
    "01177": "中国生物制药",
    "01093": "石药集团",
    "02269": "药明生物",
    "06160": "百济神州",
    "01548": "金斯瑞生物",
    "00853": "微创医疗",
    "01066": "威高股份",
    "00867": "康哲药业",
    "02196": "复旦张江",
    "00241": "阿里健康",
}

# 港股电力
HK_POWER = {
    "02638": "港灯",
    "00002": "中电控股",
    "00006": "电能实业",
    "00836": "华润电力",
    "01816": "中广核电力",
    "00135": "昆仑能源",
    "03898": "中国电力",
    "00958": "华能国际电力股份",
    "00354": "中国燃气",
    "02380": "中国电力国际发展",
}

# ============================================================
# 时间范围和质量控制
# ============================================================

START_DATE = "20100101"   # akshare格式 YYYYMMDD
END_DATE   = "20241114"
MIN_TRADING_DAYS = 2000   # 约10年数据
MAX_MISSING_RATE = 0.05
OUTPUT_DIR = "./stock_data"

# ============================================================
# 下载函数
# ============================================================

def fetch_us_stock(ticker, name, start, end):
    """下载单只美股"""
    try:
        df = ak.stock_us_hist(
            symbol=ticker,
            period="daily",
            start_date=start,
            end_date=end,
            adjust="qfq"   # 前复权
        )
        if df is None or df.empty:
            print(f"  [空数据] {ticker} {name}")
            return None

        # akshare美股列名：日期 开盘 收盘 最高 最低 成交量 成交额 振幅 涨跌幅 涨跌额 换手率
        # 统一重命名
        col_map = {
            "日期": "date", "开盘": "open", "最高": "high",
            "最低": "low",  "收盘": "close","成交量": "volume",
        }
        df = df.rename(columns=col_map)

        needed = [c for c in ["date","open","high","low","close","volume"] if c in df.columns]
        df = df[needed].copy()
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()

        return _quality_check(df, ticker, name)

    except Exception as e:
        print(f"  [失败] {ticker} {name}: {e}")
        return None


def fetch_hk_stock(symbol, name, start, end):
    """下载单只港股"""
    try:
        df = ak.stock_hk_hist(
            symbol=symbol,
            period="daily",
            start_date=start,
            end_date=end,
            adjust="qfq"
        )
        if df is None or df.empty:
            print(f"  [空数据] {symbol} {name}")
            return None

        col_map = {
            "日期": "date", "开盘": "open", "最高": "high",
            "最低": "low",  "收盘": "close","成交量": "volume",
        }
        df = df.rename(columns=col_map)

        needed = [c for c in ["date","open","high","low","close","volume"] if c in df.columns]
        df = df[needed].copy()
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()

        return _quality_check(df, symbol, name)

    except Exception as e:
        print(f"  [失败] {symbol} {name}: {e}")
        return None


def _quality_check(df, ticker, name):
    """统一质量检查"""
    total = len(df)
    if total < MIN_TRADING_DAYS:
        print(f"  [数据不足] {ticker} {name}: 仅{total}个交易日，跳过")
        return None

    missing = df.isnull().mean().max()
    if missing > MAX_MISSING_RATE:
        print(f"  [缺失率过高] {ticker} {name}: {missing:.1%}，跳过")
        return None

    df = df.ffill().dropna()
    df["ticker"] = ticker
    df["name"]   = name
    print(f"  [OK] {ticker} {name}: {total}个交易日，缺失率{missing:.2%}")
    return df


def fetch_batch(ticker_dict, label, start, end, output_dir, market="us"):
    """批量下载并保存"""
    sector_dir = os.path.join(output_dir, label)
    os.makedirs(sector_dir, exist_ok=True)

    all_dfs, success, fail = [], 0, 0

    for ticker, name in ticker_dict.items():
        if market == "us":
            df = fetch_us_stock(ticker, name, start, end)
        else:
            df = fetch_hk_stock(ticker, name, start, end)

        if df is not None:
            save_path = os.path.join(sector_dir, f"{ticker}.csv")
            df.to_csv(save_path)
            all_dfs.append(df)
            success += 1
        else:
            fail += 1

        time.sleep(1)  # 礼貌性间隔，避免请求过快

    if all_dfs:
        combined = pd.concat(all_dfs)
        combined_path = os.path.join(output_dir, f"{label}_all.csv")
        combined.to_csv(combined_path)
        print(f"\n  => 合并文件已保存: {combined_path}")

    print(f"  => {label}: 成功{success}只，失败/跳过{fail}只\n")
    return all_dfs


def print_summary(output_dir):
    print("\n" + "="*55)
    print("下载完成，文件汇总：")
    print("="*55)
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith("_all.csv"):
            fpath = os.path.join(output_dir, fname)
            try:
                df = pd.read_csv(fpath, index_col=0, parse_dates=True)
                n = df["ticker"].nunique() if "ticker" in df.columns else "?"
                print(f"  {fname:35s} | {len(df):7,}行 | {n}只")
                print(f"    日期: {df.index.min().date()} ~ {df.index.max().date()}")
            except Exception as e:
                print(f"  {fname}: 读取失败 {e}")


# ============================================================
# 主流程 - 可以注释掉不需要的板块，分批跑
# ============================================================

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"开始下载，时间范围: {START_DATE} ~ {END_DATE}\n")

    print("【1/4】美股医疗板块...")
    fetch_batch(US_MEDICAL, "us_medical", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print("【2/4】美股电力板块...")
    fetch_batch(US_POWER, "us_power", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print("【3/4】港股医疗板块...")
    fetch_batch(HK_MEDICAL, "hk_medical", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print("【4/4】港股电力板块...")
    fetch_batch(HK_POWER, "hk_power", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print_summary(OUTPUT_DIR)
    print("全部完成！")

开始下载，时间范围: 20100101 ~ 20241114

【1/4】美股医疗板块...
  [失败] JNJ 强生: HTTPSConnectionPool(host='63.push2his.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/stock/kline/get?secid=JNJ&fields1=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6&fields2=f51%2Cf52%2Cf53%2Cf54%2Cf55%2Cf56%2Cf57%2Cf58%2Cf59%2Cf60%2Cf61&klt=101&fqt=1&end=20500000&lmt=1000000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))
  [失败] PFE 辉瑞: HTTPSConnectionPool(host='63.push2his.eastmoney.com', port=443): Max retries exceeded with url: /api/qt/stock/kline/get?secid=PFE&fields1=f1%2Cf2%2Cf3%2Cf4%2Cf5%2Cf6&fields2=f51%2Cf52%2Cf53%2Cf54%2Cf55%2Cf56%2Cf57%2Cf58%2Cf59%2Cf60%2Cf61&klt=101&fqt=1&end=20500000&lmt=1000000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)')))
  [失败] UNH 联合健康: HTTPSConnectionPool(host='63.push2his.eastmoney.com', port=443): Max retries exceeded with url: 

In [7]:
"""
股票数据获取脚本 v3 - 使用Tushare（港股 + 美股）
与现有A股数据格式保持一致

安装依赖：pip install tushare pandas
"""

import tushare as ts
import pandas as pd
import os
import time

# ============================================================
# ！！！在这里填入你的Token！！！
# ============================================================
TUSHARE_TOKEN = "YOUR_TUSHARE_TOKEN"

# ============================================================
# 股票池配置
# ============================================================

# 港股医疗
HK_MEDICAL = {
    "01177.HK": "中国生物制药",
    "01093.HK": "石药集团",
    "02269.HK": "药明生物",
    "00853.HK": "微创医疗",
    "00867.HK": "康哲药业",
    "01066.HK": "威高股份",
    "00241.HK": "阿里健康",
    "02196.HK": "复旦张江",
    "01548.HK": "金斯瑞生物",
    "06160.HK": "百济神州",
}

# 港股电力
HK_POWER = {
    "02638.HK": "港灯",
    "00002.HK": "中电控股",
    "00006.HK": "电能实业",
    "00836.HK": "华润电力",
    "01816.HK": "中广核电力",
    "00135.HK": "昆仑能源",
    "03898.HK": "中国电力",
    "00958.HK": "华能国际",
    "02380.HK": "中国电力国际",
    "00354.HK": "中国燃气",
}

# 美股医疗
US_MEDICAL = {
    "AAPL":  "苹果",        # 备用，先测试接口是否通
    "JNJ":   "强生",
    "PFE":   "辉瑞",
    "UNH":   "联合健康",
    "ABT":   "雅培",
    "MRK":   "默沙东",
    "TMO":   "赛默飞",
    "DHR":   "丹纳赫",
    "MDT":   "美敦力",
    "BMY":   "百时美施贵宝",
}

# 美股电力
US_POWER = {
    "NEE":  "NextEra能源",
    "DUK":  "杜克能源",
    "SO":   "南方公司",
    "AEP":  "美国电力",
    "EXC":  "Exelon",
    "SRE":  "Sempra",
    "XEL":  "Xcel能源",
    "ES":   "Eversource",
    "WEC":  "WEC能源",
    "D":    "优势能源",
}

# ============================================================
# 时间范围和质量控制
# ============================================================

START_DATE       = "20100101"
END_DATE         = "20241114"
MIN_TRADING_DAYS = 1500    # 港股/美股交易日略少于A股，稍微放宽
MAX_MISSING_RATE = 0.05
OUTPUT_DIR       = "./stock_data_tushare"

# tushare每分钟请求限制，每次请求后等待
SLEEP_BETWEEN = 0.4   # 普通用户适当加大，如果报频率错误改成1.0

# ============================================================
# 核心函数
# ============================================================

def fetch_hk_stock(pro, ts_code, name, start, end):
    """下载单只港股日线"""
    try:
        df = pro.hk_daily(
            ts_code=ts_code,
            start_date=start,
            end_date=end,
            fields="trade_date,open,high,low,close,vol"
        )
        if df is None or df.empty:
            print(f"  [空数据] {ts_code} {name}")
            return None

        df = df.rename(columns={"trade_date": "date", "vol": "volume"})
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
        df = df[["open", "high", "low", "close", "volume"]]

        return _quality_check(df, ts_code, name)

    except Exception as e:
        print(f"  [失败] {ts_code} {name}: {e}")
        return None


def fetch_us_stock(pro, ts_code, name, start, end):
    """下载单只美股日线"""
    try:
        df = pro.us_daily(
            ts_code=ts_code,
            start_date=start,
            end_date=end,
            fields="trade_date,open,high,low,close,vol"
        )
        if df is None or df.empty:
            print(f"  [空数据] {ts_code} {name}")
            return None

        df = df.rename(columns={"trade_date": "date", "vol": "volume"})
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
        df = df[["open", "high", "low", "close", "volume"]]

        return _quality_check(df, ts_code, name)

    except Exception as e:
        print(f"  [失败] {ts_code} {name}: {e}")
        return None


def _quality_check(df, ticker, name):
    """统一质量检查"""
    total = len(df)
    if total < MIN_TRADING_DAYS:
        print(f"  [数据不足] {ticker} {name}: 仅{total}个交易日，跳过")
        return None

    missing = df.isnull().mean().max()
    if missing > MAX_MISSING_RATE:
        print(f"  [缺失率过高] {ticker} {name}: {missing:.1%}，跳过")
        return None

    df = df.ffill().dropna()
    df["ticker"] = ticker
    df["name"]   = name
    print(f"  [OK] {ticker} {name}: {total}个交易日，缺失率{missing:.2%}")
    return df


def fetch_batch(pro, ticker_dict, label, start, end, output_dir, market="hk"):
    """批量下载并保存CSV"""
    sector_dir = os.path.join(output_dir, label)
    os.makedirs(sector_dir, exist_ok=True)

    all_dfs, success, fail = [], 0, 0

    for ticker, name in ticker_dict.items():
        if market == "hk":
            df = fetch_hk_stock(pro, ticker, name, start, end)
        else:
            df = fetch_us_stock(pro, ticker, name, start, end)

        if df is not None:
            safe_name = ticker.replace(".", "_")
            save_path  = os.path.join(sector_dir, f"{safe_name}.csv")
            df.to_csv(save_path)
            all_dfs.append(df)
            success += 1
        else:
            fail += 1

        time.sleep(SLEEP_BETWEEN)

    if all_dfs:
        combined      = pd.concat(all_dfs)
        combined_path = os.path.join(output_dir, f"{label}_all.csv")
        combined.to_csv(combined_path)
        print(f"\n  => 合并文件已保存: {combined_path}")

    print(f"  => {label}: 成功{success}只，失败/跳过{fail}只\n")
    return all_dfs


def print_summary(output_dir):
    print("\n" + "="*55)
    print("下载完成，文件汇总：")
    print("="*55)
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith("_all.csv"):
            fpath = os.path.join(output_dir, fname)
            try:
                df = pd.read_csv(fpath, index_col=0, parse_dates=True)
                n  = df["ticker"].nunique() if "ticker" in df.columns else "?"
                print(f"  {fname:40s} | {len(df):7,}行 | {n}只")
                print(f"    日期: {df.index.min().date()} ~ {df.index.max().date()}")
            except Exception as e:
                print(f"  {fname}: 读取失败 {e}")


# ============================================================
# 主流程
# ============================================================

if __name__ == "__main__":
    # 初始化tushare
    ts.set_token(TUSHARE_TOKEN)
    pro = ts.pro_api()

    # 先快速验证token和权限
    print("验证Token中...")
    try:
        info = pro.query('user', fields='nick_name,account_type,remaining_count')
        print(f"  账号：{info.iloc[0]['nick_name']}")
        print(f"  类型：{info.iloc[0]['account_type']}")
        print(f"  剩余积分：{info.iloc[0]['remaining_count']}\n")
    except Exception as e:
        print(f"  Token验证失败: {e}")
        print("  请检查token是否正确，或网络是否连通tushare.pro")
        exit(1)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"开始下载，时间范围: {START_DATE} ~ {END_DATE}\n")

    print("【1/4】港股医疗板块...")
    fetch_batch(pro, HK_MEDICAL, "hk_medical", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print("【2/4】港股电力板块...")
    fetch_batch(pro, HK_POWER, "hk_power", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print("【3/4】美股医疗板块...")
    fetch_batch(pro, US_MEDICAL, "us_medical", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print("【4/4】美股电力板块...")
    fetch_batch(pro, US_POWER, "us_power", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print_summary(OUTPUT_DIR)
    print("全部完成！")

验证Token中...
  Token验证失败: 请指定正确的接口名
  请检查token是否正确，或网络是否连通tushare.pro
开始下载，时间范围: 20100101 ~ 20241114

【1/4】港股医疗板块...
  [失败] 01177.HK 中国生物制药: 抱歉，您访问接口(hk_daily)频率超限(1次/小时)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01093.HK 石药集团: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 02269.HK 药明生物: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00853.HK 微创医疗: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00867.HK 康哲药业: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01066.HK 威高股份: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00241.HK 阿里健康: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 02196.HK 复旦张江: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01548.HK 金斯瑞生物: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https:

In [7]:
"""
股票数据获取脚本 v3 - 使用Tushare（港股 + 美股）
与现有A股数据格式保持一致

安装依赖：pip install tushare pandas
"""

import tushare as ts
import pandas as pd
import os
import time

# ============================================================
# ！！！在这里填入你的Token！！！
# ============================================================
TUSHARE_TOKEN = "YOUR_TUSHARE_TOKEN"

# ============================================================
# 股票池配置
# ============================================================

# 港股医疗
HK_MEDICAL = {
    "01177.HK": "中国生物制药",
    "01093.HK": "石药集团",
    "02269.HK": "药明生物",
    "00853.HK": "微创医疗",
    "00867.HK": "康哲药业",
    "01066.HK": "威高股份",
    "00241.HK": "阿里健康",
    "02196.HK": "复旦张江",
    "01548.HK": "金斯瑞生物",
    "06160.HK": "百济神州",
}

# 港股电力
HK_POWER = {
    "02638.HK": "港灯",
    "00002.HK": "中电控股",
    "00006.HK": "电能实业",
    "00836.HK": "华润电力",
    "01816.HK": "中广核电力",
    "00135.HK": "昆仑能源",
    "03898.HK": "中国电力",
    "00958.HK": "华能国际",
    "02380.HK": "中国电力国际",
    "00354.HK": "中国燃气",
}

# 美股医疗
# US_MEDICAL = {
#     "AAPL":  "苹果",        # 备用，先测试接口是否通
#     "JNJ":   "强生",
#     "PFE":   "辉瑞",
#     "UNH":   "联合健康",
#     "ABT":   "雅培",
#     "MRK":   "默沙东",
#     "TMO":   "赛默飞",
#     "DHR":   "丹纳赫",
#     "MDT":   "美敦力",
#     "BMY":   "百时美施贵宝",
# }

# # 美股电力
# US_POWER = {
#     "NEE":  "NextEra能源",
#     "DUK":  "杜克能源",
#     "SO":   "南方公司",
#     "AEP":  "美国电力",
#     "EXC":  "Exelon",
#     "SRE":  "Sempra",
#     "XEL":  "Xcel能源",
#     "ES":   "Eversource",
#     "WEC":  "WEC能源",
#     "D":    "优势能源",
# }

# 美股医疗 / Healthcare，20只
US_MEDICAL = {
    "JNJ":  "Johnson & Johnson",
    "PFE":  "Pfizer",
    "UNH":  "UnitedHealth",
    "ABT":  "Abbott",
    "MRK":  "Merck",
    "TMO":  "Thermo Fisher",
    "DHR":  "Danaher",
    "MDT":  "Medtronic",
    "BMY":  "Bristol Myers Squibb",
    "AMGN": "Amgen",
    "GILD": "Gilead Sciences",
    "LLY":  "Eli Lilly",
    "ABBV": "AbbVie",
    "ISRG": "Intuitive Surgical",
    "SYK":  "Stryker",
    "BSX":  "Boston Scientific",
    "CI":   "Cigna",
    "HUM":  "Humana",
    "REGN": "Regeneron",
    "VRTX": "Vertex Pharmaceuticals",
}

# 美股电力 / Utilities，20只
US_POWER = {
    "NEE": "NextEra Energy",
    "DUK": "Duke Energy",
    "SO":  "Southern Company",
    "AEP": "American Electric Power",
    "EXC": "Exelon",
    "SRE": "Sempra",
    "XEL": "Xcel Energy",
    "ES":  "Eversource Energy",
    "WEC": "WEC Energy",
    "D":   "Dominion Energy",
    "PEG": "Public Service Enterprise Group",
    "ED":  "Consolidated Edison",
    "EIX": "Edison International",
    "FE":  "FirstEnergy",
    "ETR": "Entergy",
    "CMS": "CMS Energy",
    "CNP": "CenterPoint Energy",
    "ATO": "Atmos Energy",
    "LNT": "Alliant Energy",
    "NI":  "NiSource",
}

# ============================================================
# 时间范围和质量控制
# ============================================================

START_DATE       = "20100101"
END_DATE         = "20251114"
MIN_TRADING_DAYS = 1500    # 港股/美股交易日略少于A股，稍微放宽
MAX_MISSING_RATE = 0.05
OUTPUT_DIR       = "./stock_data_tushare"

# tushare每分钟请求限制，每次请求后等待
SLEEP_BETWEEN = 0.4   # 普通用户适当加大，如果报频率错误改成1.0

# ============================================================
# 核心函数
# ============================================================

def fetch_hk_stock(pro, ts_code, name, start, end):
    """下载单只港股日线"""
    try:
        df = pro.hk_daily(
            ts_code=ts_code,
            start_date=start,
            end_date=end,
            fields="trade_date,open,high,low,close,vol"
        )
        if df is None or df.empty:
            print(f"  [空数据] {ts_code} {name}")
            return None

        df = df.rename(columns={"trade_date": "date", "vol": "volume"})
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
        df = df[["open", "high", "low", "close", "volume"]]

        return _quality_check(df, ts_code, name)

    except Exception as e:
        print(f"  [失败] {ts_code} {name}: {e}")
        return None


def fetch_us_stock(pro, ts_code, name, start, end):
    """下载单只美股日线"""
    try:
        df = pro.us_daily(
            ts_code=ts_code,
            start_date=start,
            end_date=end,
            fields="trade_date,open,high,low,close,vol"
        )
        if df is None or df.empty:
            print(f"  [空数据] {ts_code} {name}")
            return None

        df = df.rename(columns={"trade_date": "date", "vol": "volume"})
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date").sort_index()
        df = df[["open", "high", "low", "close", "volume"]]

        return _quality_check(df, ts_code, name)

    except Exception as e:
        print(f"  [失败] {ts_code} {name}: {e}")
        return None


def _quality_check(df, ticker, name):
    """统一质量检查"""
    total = len(df)
    if total < MIN_TRADING_DAYS:
        print(f"  [数据不足] {ticker} {name}: 仅{total}个交易日，跳过")
        return None

    missing = df.isnull().mean().max()
    if missing > MAX_MISSING_RATE:
        print(f"  [缺失率过高] {ticker} {name}: {missing:.1%}，跳过")
        return None

    df = df.ffill().dropna()
    df["ticker"] = ticker
    df["name"]   = name
    print(f"  [OK] {ticker} {name}: {total}个交易日，缺失率{missing:.2%}")
    return df


def fetch_batch(pro, ticker_dict, label, start, end, output_dir, market="hk"):
    """批量下载并保存CSV"""
    sector_dir = os.path.join(output_dir, label)
    os.makedirs(sector_dir, exist_ok=True)

    all_dfs, success, fail = [], 0, 0

    for ticker, name in ticker_dict.items():
        if market == "hk":
            df = fetch_hk_stock(pro, ticker, name, start, end)
        else:
            df = fetch_us_stock(pro, ticker, name, start, end)

        if df is not None:
            safe_name = ticker.replace(".", "_")
            save_path  = os.path.join(sector_dir, f"{safe_name}.csv")
            df.to_csv(save_path)
            all_dfs.append(df)
            success += 1
        else:
            fail += 1

        time.sleep(SLEEP_BETWEEN)

    if all_dfs:
        combined      = pd.concat(all_dfs)
        combined_path = os.path.join(output_dir, f"{label}_all.csv")
        combined.to_csv(combined_path)
        print(f"\n  => 合并文件已保存: {combined_path}")

    print(f"  => {label}: 成功{success}只，失败/跳过{fail}只\n")
    return all_dfs


def print_summary(output_dir):
    print("\n" + "="*55)
    print("下载完成，文件汇总：")
    print("="*55)
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith("_all.csv"):
            fpath = os.path.join(output_dir, fname)
            try:
                df = pd.read_csv(fpath, index_col=0, parse_dates=True)
                n  = df["ticker"].nunique() if "ticker" in df.columns else "?"
                print(f"  {fname:40s} | {len(df):7,}行 | {n}只")
                print(f"    日期: {df.index.min().date()} ~ {df.index.max().date()}")
            except Exception as e:
                print(f"  {fname}: 读取失败 {e}")


# ============================================================
# 主流程
# ============================================================

if __name__ == "__main__":
    # 初始化tushare
    ts.set_token(TUSHARE_TOKEN)
    pro = ts.pro_api()

    # 先快速验证token和权限
    print("验证Token中...")
    try:
        info = pro.query('user', fields='nick_name,account_type,remaining_count')
        print(f"  账号：{info.iloc[0]['nick_name']}")
        print(f"  类型：{info.iloc[0]['account_type']}")
        print(f"  剩余积分：{info.iloc[0]['remaining_count']}\n")
    except Exception as e:
        print(f"  Token验证失败: {e}")
        print("  请检查token是否正确，或网络是否连通tushare.pro")
        exit(1)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"开始下载，时间范围: {START_DATE} ~ {END_DATE}\n")

    print("【1/4】港股医疗板块...")
    fetch_batch(pro, HK_MEDICAL, "hk_medical", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print("【2/4】港股电力板块...")
    fetch_batch(pro, HK_POWER, "hk_power", START_DATE, END_DATE, OUTPUT_DIR, market="hk")

    print("【3/4】美股医疗板块...")
    fetch_batch(pro, US_MEDICAL, "us_medical", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print("【4/4】美股电力板块...")
    fetch_batch(pro, US_POWER, "us_power", START_DATE, END_DATE, OUTPUT_DIR, market="us")

    print_summary(OUTPUT_DIR)
    print("全部完成！")

验证Token中...
  Token验证失败: 请指定正确的接口名
  请检查token是否正确，或网络是否连通tushare.pro
开始下载，时间范围: 20100101 ~ 20251114

【1/4】港股医疗板块...
  [失败] 01177.HK 中国生物制药: 抱歉，您访问接口(hk_daily)频率超限(1次/小时)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01093.HK 石药集团: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 02269.HK 药明生物: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00853.HK 微创医疗: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00867.HK 康哲药业: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01066.HK 威高股份: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 00241.HK 阿里健康: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 02196.HK 复旦张江: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https://tushare.pro/document/1?doc_id=108。
  [失败] 01548.HK 金斯瑞生物: 抱歉，您访问接口(hk_daily)频率超限(1次/分钟)，具体频次详情：https:

In [ ]:
---------------------------------特征工程----------------------------

In [11]:
"""
美股完整特征工程脚本
特征池与A股完全对齐（基于A股特征工程代码）

输入：stock_data_tushare/us_medical_all.csv
      stock_data_tushare/us_power_all.csv
输出：stock_data_tushare/us_medical_features_full.csv
      stock_data_tushare/us_power_features_full.csv

安装依赖：pip install pandas numpy
"""

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 路径配置
# ============================================================
INPUT_DIR  = "./stock_data_tushare"
OUTPUT_DIR = "./stock_data_tushare"

FILES = {
    "us_medical": "us_medical_all.csv",
    "us_power":   "us_power_all.csv",
}

# ============================================================
# 美股流通股本（用于计算换手率，单位：股）
# 注：为当前近似值，历史期间因回购/增发会有偏差
# ============================================================
US_SHARES_OUTSTANDING = {
    # 医疗板块
    "JNJ":  4.05e9, "PFE":  5.65e9, "UNH":  0.92e9, "ABT":  1.74e9,
    "MRK":  2.53e9, "TMO":  0.38e9, "DHR":  0.74e9, "MDT":  1.28e9,
    "BMY":  2.03e9, "AMGN": 0.54e9, "GILD": 1.25e9, "LLY":  0.95e9,
    "ABBV": 1.77e9, "ISRG": 0.36e9, "SYK":  0.38e9, "BSX":  1.47e9,
    "CI":   0.28e9, "HUM":  0.12e9, "REGN": 0.11e9, "VRTX": 0.26e9,
    # 电力板块
    "NEE":  2.06e9, "DUK":  0.77e9, "SO":   1.10e9, "AEP":  0.53e9,
    "EXC":  1.00e9, "SRE":  0.64e9, "XEL":  0.57e9, "ES":   0.36e9,
    "WEC":  0.32e9, "D":    0.84e9, "PEG":  0.50e9, "ED":   0.35e9,
    "EIX":  0.39e9, "FE":   0.58e9, "ETR":  0.21e9, "CMS":  0.30e9,
    "CNP":  0.64e9, "ATO":  0.15e9, "LNT":  0.26e9, "NI":   0.45e9,
}

# ============================================================
# 原始价格特征（对齐A股，pre_close用close.shift(1)替代）
# ============================================================
ORIGINAL_FEATURE_COLS = [
    'open', 'close', 'high', 'low', 'vol', 'amount',
    'turnover_rate', 'pre_close',
]

# ============================================================
# 技术指标特征（与A股完全一致）
# ============================================================
TECHNICAL_FEATURE_COLS = [
    'ma5', 'ma10', 'ma20', 'ma60',
    'close_to_ma5', 'close_to_ma20',
    'volume_ma5', 'volume_ma10', 'volume_ratio',
    'roc_5', 'roc_10', 'roc_20',
    'volatility_5', 'volatility_10', 'volatility_20',
    'high_low_range', 'close_open_range',
    'rsi',
    'macd', 'macd_signal', 'macd_hist',
    'bb_upper', 'bb_lower', 'bb_middle', 'bb_width', 'bb_position',
    'atr',
    'momentum_5', 'momentum_10',
]

# ============================================================
# 时间特征（与A股完全一致）
# ============================================================
TIME_FEATURE_COLS = [
    'year', 'month', 'day', 'dayofweek', 'quarter',
    'dayofyear', 'weekofyear',
    'month_sin', 'month_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'dayofyear_sin', 'dayofyear_cos',
]

ALL_FEATURE_COLS = ORIGINAL_FEATURE_COLS + TECHNICAL_FEATURE_COLS

# ============================================================
# 单只股票特征构造（对齐A股逻辑）
# ============================================================
def calculate_features_one_stock(group):
    """
    为单只美股计算全部特征，逻辑与A股完全对齐
    关键：所有原始价格特征和技术指标均执行shift(1)
    目标变量：下一日涨跌幅（pct_chg_next）
    """
    group = group.sort_values('trade_date').reset_index(drop=True)

    # ---------- 数据清洗 ----------
    group = group[
        (group['open']  > 0) & (group['high'] > 0) &
        (group['low']   > 0) & (group['close'] > 0)
    ]
    group = group[group['high'] >= group['low']]
    group = group[group['vol']  > 0]

    group['pct_chg_raw'] = group['close'].pct_change() * 100
    group = group[group['pct_chg_raw'].abs() < 100].copy()
    group = group.reset_index(drop=True)

    # ---------- 交易日间隔（对齐A股） ----------
    group['days_since_last_trade'] = group['trade_date'].diff().dt.days
    group.loc[0, 'days_since_last_trade'] = 1

    # ---------- 基础衍生（对齐A股原始字段） ----------
    group['amount']    = group['close'] * group['vol']
    ticker             = group['stock_code'].iloc[0]
    shares             = US_SHARES_OUTSTANDING.get(ticker, np.nan)
    group['turnover_rate'] = (
        group['vol'] / shares * 100 if (not isinstance(shares, float) or not np.isnan(shares))
        else np.nan
    )
    # pre_close：用前一日收盘价替代（对齐A股字段）
    group['pre_close'] = group['close'].shift(1)

    # ---------- 均线 ----------
    group['ma5']  = group['close'].rolling(window=5,  min_periods=1).mean()
    group['ma10'] = group['close'].rolling(window=10, min_periods=1).mean()
    group['ma20'] = group['close'].rolling(window=20, min_periods=1).mean()
    group['ma60'] = group['close'].rolling(window=60, min_periods=1).mean()

    # ---------- 价格相对位置 ----------
    group['close_to_ma5']  = (group['close'] - group['ma5'])  / (group['ma5']  + 1e-8)
    group['close_to_ma20'] = (group['close'] - group['ma20']) / (group['ma20'] + 1e-8)

    # ---------- 成交量指标 ----------
    group['volume_ma5']   = group['vol'].rolling(window=5,  min_periods=1).mean()
    group['volume_ma10']  = group['vol'].rolling(window=10, min_periods=1).mean()
    group['volume_ratio'] = group['vol'] / (group['volume_ma5'] + 1e-8)

    # ---------- 变化率 ----------
    group['roc_5']  = group['close'].pct_change(periods=5)  * 100
    group['roc_10'] = group['close'].pct_change(periods=10) * 100
    group['roc_20'] = group['close'].pct_change(periods=20) * 100

    # ---------- 波动率 ----------
    group['volatility_5']  = group['close'].rolling(window=5,  min_periods=1).std()
    group['volatility_10'] = group['close'].rolling(window=10, min_periods=1).std()
    group['volatility_20'] = group['close'].rolling(window=20, min_periods=1).std()

    # ---------- 价格区间 ----------
    group['high_low_range']   = (group['high'] - group['low'])    / (group['close'] + 1e-8)
    group['close_open_range'] = (group['close'] - group['open'])  / (group['open']  + 1e-8)

    # ---------- RSI ----------
    delta = group['close'].diff()
    gain  = (delta.where(delta > 0, 0)).rolling(window=14, min_periods=1).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(window=14, min_periods=1).mean()
    rs    = gain / (loss + 1e-8)
    group['rsi'] = 100 - (100 / (1 + rs))

    # ---------- MACD ----------
    ema12              = group['close'].ewm(span=12, adjust=False).mean()
    ema26              = group['close'].ewm(span=26, adjust=False).mean()
    group['macd']      = ema12 - ema26
    group['macd_signal'] = group['macd'].ewm(span=9, adjust=False).mean()
    group['macd_hist'] = group['macd'] - group['macd_signal']

    # ---------- 布林带 ----------
    bb_std               = group['close'].rolling(window=20, min_periods=1).std()
    group['bb_middle']   = group['close'].rolling(window=20, min_periods=1).mean()
    group['bb_upper']    = group['bb_middle'] + (bb_std * 2)
    group['bb_lower']    = group['bb_middle'] - (bb_std * 2)
    group['bb_width']    = (group['bb_upper'] - group['bb_lower']) / (group['bb_middle'] + 1e-8)
    group['bb_position'] = (
        (group['close'] - group['bb_lower']) /
        (group['bb_upper'] - group['bb_lower'] + 1e-8)
    )

    # ---------- ATR ----------
    high_low    = group['high'] - group['low']
    high_close  = (group['high'] - group['close'].shift(1)).abs()
    low_close   = (group['low']  - group['close'].shift(1)).abs()
    true_range  = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    group['atr'] = true_range.rolling(window=14, min_periods=1).mean()

    # ---------- 动量 ----------
    group['momentum_5']  = group['close'] - group['close'].shift(5)
    group['momentum_10'] = group['close'] - group['close'].shift(10)

    # ---------- 时间特征（不需要shift，直接从日期提取） ----------
    group['year']       = group['trade_date'].dt.year
    group['month']      = group['trade_date'].dt.month
    group['day']        = group['trade_date'].dt.day
    group['dayofweek']  = group['trade_date'].dt.dayofweek
    group['quarter']    = group['trade_date'].dt.quarter
    group['dayofyear']  = group['trade_date'].dt.dayofyear
    group['weekofyear'] = group['trade_date'].dt.isocalendar().week.astype(int)

    group['month_sin']      = np.sin(2 * np.pi * group['month']     / 12)
    group['month_cos']      = np.cos(2 * np.pi * group['month']     / 12)
    group['dayofweek_sin']  = np.sin(2 * np.pi * group['dayofweek'] / 7)
    group['dayofweek_cos']  = np.cos(2 * np.pi * group['dayofweek'] / 7)
    group['dayofyear_sin']  = np.sin(2 * np.pi * group['dayofyear'] / 365)
    group['dayofyear_cos']  = np.cos(2 * np.pi * group['dayofyear'] / 365)

    # ---------- 目标变量：下一日涨跌幅 ----------
    group['pct_chg_next'] = group['pct_chg_raw'].shift(-1)

    # ---------- 所有价格/技术特征滞后一期（防止数据泄露） ----------
    group[ALL_FEATURE_COLS] = group[ALL_FEATURE_COLS].shift(1)

    # ---------- 长停牌标记 ----------
    group['is_after_long_halt'] = (group['days_since_last_trade'] > 10).astype(int)

    # ---------- 按股票前向填充 ----------
    group[ALL_FEATURE_COLS] = group[ALL_FEATURE_COLS].ffill()

    # ---------- 去除缺失 ----------
    drop_cols = ALL_FEATURE_COLS + TIME_FEATURE_COLS + ['pct_chg_next']
    group = group.dropna(subset=drop_cols).reset_index(drop=True)

    # ---------- 清理中间列 ----------
    group = group.drop(columns=['pct_chg_raw'], errors='ignore')

    # ---------- 保留列顺序对齐A股 ----------
    keep = (
        ['stock_code', 'trade_date'] +
        ORIGINAL_FEATURE_COLS +
        TECHNICAL_FEATURE_COLS +
        TIME_FEATURE_COLS +
        ['days_since_last_trade', 'is_after_long_halt', 'pct_chg_next']
    )
    keep = [c for c in keep if c in group.columns]
    return group[keep]

# ============================================================
# 主流程
# ============================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for label, fname in FILES.items():
        path = os.path.join(INPUT_DIR, fname)
        print(f"\n{'='*60}")
        print(f"处理: {label}  ({fname})")
        print(f"{'='*60}")

        df = pd.read_csv(path)

        # 统一列名对齐A股
        df = df.rename(columns={
            "date":   "trade_date",
            "ticker": "stock_code",
            "volume": "vol",
        })

        df["trade_date"] = pd.to_datetime(df["trade_date"])
        df["stock_code"] = df["stock_code"].astype(str)

        # 逐只股票处理
        out_list = []
        for ticker, g in df.groupby("stock_code"):
            g = g.copy()
            result = calculate_features_one_stock(g)
            out_list.append(result)
            print(f"  [{ticker}] {len(result)}行")

        out = pd.concat(out_list, ignore_index=True)

        # 保存
        save_path = os.path.join(OUTPUT_DIR, f"{label}_features_full.csv")
        out.to_csv(save_path, index=False, encoding="utf-8-sig")

        # 汇总
        print(f"\n保存至: {save_path}")
        print(f"shape:        {out.shape}")
        print(f"日期:         {out['trade_date'].min()} ~ {out['trade_date'].max()}")
        print(f"股票数:       {out['stock_code'].nunique()}")
        print(f"原始价格特征: {len(ORIGINAL_FEATURE_COLS)}")
        print(f"技术指标特征: {len(TECHNICAL_FEATURE_COLS)}")
        print(f"时间特征:     {len(TIME_FEATURE_COLS)}")
        print(f"总特征数:     {len(ORIGINAL_FEATURE_COLS)+len(TECHNICAL_FEATURE_COLS)+len(TIME_FEATURE_COLS)}")
        missing = out[ALL_FEATURE_COLS + TIME_FEATURE_COLS + ['pct_chg_next']].isna().sum().sum()
        print(f"缺失值:       {missing}")
        print(f"\n完整列名:")
        print(list(out.columns))


处理: us_medical  (us_medical_all.csv)
  [ABBV] 3231行
  [ABT] 3970行
  [AMGN] 3970行
  [BMY] 3970行
  [BSX] 3970行
  [CI] 3970行
  [DHR] 3970行
  [GILD] 3970行
  [HUM] 3970行
  [ISRG] 3970行
  [JNJ] 3970行
  [LLY] 3970行
  [MDT] 3970行
  [MRK] 3970行
  [PFE] 3970行
  [REGN] 3970行
  [SYK] 3970行
  [TMO] 3970行
  [UNH] 3970行
  [VRTX] 3970行

保存至: ./stock_data_tushare\us_medical_features_full.csv
shape:        (78661, 55)
日期:         2010-02-04 00:00:00 ~ 2025-11-13 00:00:00
股票数:       20
原始价格特征: 8
技术指标特征: 29
时间特征:     13
总特征数:     50
缺失值:       0

完整列名:
['stock_code', 'trade_date', 'open', 'close', 'high', 'low', 'vol', 'amount', 'turnover_rate', 'pre_close', 'ma5', 'ma10', 'ma20', 'ma60', 'close_to_ma5', 'close_to_ma20', 'volume_ma5', 'volume_ma10', 'volume_ratio', 'roc_5', 'roc_10', 'roc_20', 'volatility_5', 'volatility_10', 'volatility_20', 'high_low_range', 'close_open_range', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_lower', 'bb_middle', 'bb_width', 'bb_position', 'atr', 'momentum_5'

In [8]:
"""
美股特征选择脚本
基于A股特征选择代码，去掉石油特征，对齐美股数据格式

输入：stock_data_tushare/us_medical_features_full.csv
      stock_data_tushare/us_power_features_full.csv
输出：stock_data_tushare/feature_selection/

安装依赖：pip install pandas numpy scikit-learn lightgbm xgboost matplotlib seaborn
"""

import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from datetime import datetime
import json
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False

print("\n" + "=" * 80)
print("美股特征选择（对齐A股逻辑，无石油特征）")
print("=" * 80)

# ============================================================
# 配置
# ============================================================
INPUT_DIR  = "./stock_data_tushare"
OUTPUT_DIR = "./stock_data_tushare/feature_selection"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("./visualizations", exist_ok=True)

TRAIN_END_DATE = '2022-12-31'
TARGET_COL     = 'pct_chg_next'

FILES = {
    "us_medical": "us_medical_features_full.csv",
    "us_power":   "us_power_features_full.csv",
}

# ============================================================
# 特征类别定义（去掉石油特征，对齐美股）
# ============================================================
original_features = [
    'open', 'close', 'high', 'low', 'vol', 'amount',
    'turnover_rate', 'pre_close'
]

time_features_numeric = [
    'year', 'month', 'day', 'dayofweek', 'quarter',
    'dayofyear', 'weekofyear'
]

time_features_cyclic = [
    'month_sin', 'month_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'dayofyear_sin', 'dayofyear_cos',
]

technical_features = [
    'ma5', 'ma10', 'ma20', 'ma60',
    'close_to_ma5', 'close_to_ma20',
    'volume_ma5', 'volume_ma10', 'volume_ratio',
    'roc_5', 'roc_10', 'roc_20',
    'volatility_5', 'volatility_10', 'volatility_20',
    'high_low_range', 'close_open_range',
    'rsi',
    'macd', 'macd_signal', 'macd_hist',
    'bb_middle', 'bb_upper', 'bb_lower', 'bb_width', 'bb_position',
    'atr',
    'momentum_5', 'momentum_10',
]

# 候选特征（无石油特征）
candidate_features = original_features + time_features_numeric + time_features_cyclic + technical_features

print(f"\n特征构成:")
print(f"  原始市场特征:          {len(original_features)}")
print(f"  时间特征（周期性编码）: {len(time_features_cyclic)}")
print(f"  技术指标特征:          {len(technical_features)}")
print(f"  候选特征总数:          {len(candidate_features)}")

# ============================================================
# JSON编码器（处理numpy类型）
# ============================================================
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        if isinstance(obj, np.bool_):    return bool(obj)
        return super().default(obj)

# ============================================================
# 特征选择主函数
# ============================================================
def feature_selection_pipeline(df, dataset_name, feature_cols,
                                target_col=TARGET_COL,
                                train_end_date=TRAIN_END_DATE,
                                # redundancy_threshold=0.95,
                               redundancy_threshold=1.0,
                                top_k=50,
                                verbose=True):

    if verbose:
        print(f"\n{'='*70}")
        print(f"特征选择: {dataset_name}")
        print(f"{'='*70}")

    # ------ 步骤0：目标泄露检查 ------
    leakage = [c for c in feature_cols if c in [target_col, 'pct_chg', 'change']]
    if leakage:
        raise ValueError(f"检测到目标泄露: {leakage}")
    if verbose:
        print(f"\n[步骤0] 目标泄露检查: ✅ 通过")

    # ------ 步骤1：时间隔离，只用训练集 ------
    df_train = df[df['trade_date'] <= pd.to_datetime(train_end_date)].copy()
    if verbose:
        print(f"\n[步骤1] 训练集: {len(df_train):,}行 | "
              f"{df_train['trade_date'].min().date()} ~ {df_train['trade_date'].max().date()} | "
              f"{df_train['stock_code'].nunique()}只股票")

    # ------ 步骤2：构造特征矩阵和目标 ------
    df_train = df_train.sort_values(['stock_code', 'trade_date']).reset_index(drop=True)

    # 只保留feature_cols中在df里存在的列
    valid_cols = [c for c in feature_cols if c in df_train.columns]
    missing_cols = [c for c in feature_cols if c not in df_train.columns]
    if missing_cols and verbose:
        print(f"  ⚠️  以下特征不在数据中，已跳过: {missing_cols}")

    valid_mask = df_train[target_col].notna()
    df_train = df_train[valid_mask].copy()

    X = df_train[valid_cols].copy()
    y = df_train[target_col].copy()

    if verbose:
        print(f"\n[步骤2] 特征矩阵: {X.shape} | 目标: {y.shape}")

    # ------ 步骤3：缺失值处理（按股票分组） ------
    X = X.replace([np.inf, -np.inf], np.nan)
    missing_before = X.isna().sum().sum()

    if missing_before > 0:
        X_filled = []
        for stock, grp in X.groupby(df_train['stock_code']):
            grp = grp.ffill().fillna(0)
            X_filled.append(grp)
        X = pd.concat(X_filled).loc[X.index]

    if verbose:
        print(f"\n[步骤3] 缺失值: {missing_before} → {X.isna().sum().sum()}")

    X_values = X.values
    y_values = y.values

    # ------ 步骤4：多方法特征重要性 ------
    if verbose:
        print(f"\n[步骤4] 计算特征重要性...")

    print("  [1/6] LightGBM...")
    lgb_imp = lgb.LGBMRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.05,
        random_state=42, verbose=-1, n_jobs=-1
    ).fit(X_values, y_values).feature_importances_

    print("  [2/6] XGBoost...")
    xgb_imp = xgb.XGBRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.05,
        random_state=42, tree_method='hist', verbosity=0, n_jobs=-1
    ).fit(X_values, y_values).feature_importances_

    print("  [3/6] Random Forest...")
    rf_imp = RandomForestRegressor(
        n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
    ).fit(X_values, y_values).feature_importances_

    print("  [4/6] F-statistic...")
    f_scores, _ = f_regression(X_values, y_values)
    f_scores = np.nan_to_num(f_scores)

    print("  [5/6] Mutual Information...")
    mi_scores = mutual_info_regression(X_values, y_values, random_state=42)

    print("  [6/6] Pearson Correlation...")
    corr_scores = np.array([
        abs(np.corrcoef(X_values[:, i], y_values)[0, 1])
        for i in range(X_values.shape[1])
    ])
    corr_scores = np.nan_to_num(corr_scores)

    # ------ 步骤5：相关性分析 ------
    corr_matrix = np.nan_to_num(np.corrcoef(X_values.T))
    high_corr_pairs = []
    for i in range(len(valid_cols)):
        for j in range(i+1, len(valid_cols)):
            if abs(corr_matrix[i, j]) > 0.9:
                high_corr_pairs.append({
                    'Feature1': valid_cols[i],
                    'Feature2': valid_cols[j],
                    'Correlation': corr_matrix[i, j]
                })
    if verbose:
        print(f"\n[步骤5] 高度相关特征对 (|r|>0.9): {len(high_corr_pairs)}")

    # ------ 步骤6：集成评分 ------
    def norm(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn + 1e-10)

    importance_df = pd.DataFrame({
        'Feature':      valid_cols,
        'LightGBM':     norm(lgb_imp),
        'XGBoost':      norm(xgb_imp),
        'RandomForest': norm(rf_imp),
        'F_Statistic':  norm(f_scores),
        'Mutual_Info':  norm(mi_scores),
        'Target_Corr':  norm(corr_scores),
    })
    importance_df['Ensemble_Score'] = importance_df[[
        'LightGBM','XGBoost','RandomForest',
        'F_Statistic','Mutual_Info','Target_Corr'
    ]].mean(axis=1)
    importance_df['Rank'] = importance_df['Ensemble_Score'].rank(ascending=False).astype(int)
    importance_df = importance_df.sort_values('Ensemble_Score', ascending=False).reset_index(drop=True)

    # ------ 步骤7：特征筛选 ------
    top_features = importance_df.head(top_k)['Feature'].tolist()

    # 去冗余
    redundant = set()
    for pair in high_corr_pairs:
        f1, f2 = pair['Feature1'], pair['Feature2']
        if abs(pair['Correlation']) > redundancy_threshold:
            if f1 in top_features and f2 in top_features:
                s1 = importance_df[importance_df['Feature']==f1]['Ensemble_Score'].values[0]
                s2 = importance_df[importance_df['Feature']==f2]['Ensemble_Score'].values[0]
                redundant.add(f2 if s1 > s2 else f1)

    selected = [f for f in top_features if f not in redundant]

    if verbose:
        print(f"\n[步骤7] 筛选结果:")
        print(f"  候选特征: {len(valid_cols)}")
        print(f"  Top-K初选: {top_k}")
        print(f"  冗余移除: {len(redundant)}")
        print(f"  最终选定: {len(selected)}")
        print(f"\n  Top-20特征:")
        print(f"  {'排名':<5} {'状态':<10} {'特征名':<25} {'得分':<10}")
        print(f"  {'-'*52}")
        for _, row in importance_df.head(20).iterrows():
            status = "✓ 保留" if row['Feature'] in selected else "✗ 移除"
            print(f"  {row['Rank']:>4}  {status:<10} {row['Feature']:<25} {row['Ensemble_Score']:.4f}")

    metadata = {
        'dataset_name':        dataset_name,
        'train_end_date':      train_end_date,
        'n_train_samples':     len(X_values),
        'n_candidate_features':len(valid_cols),
        'top_k':               top_k,
        'redundancy_threshold':redundancy_threshold,
        'n_redundant_removed': len(redundant),
        'n_final_selected':    len(selected),
        'timestamp':           datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    return selected, importance_df, high_corr_pairs, metadata

# ============================================================
# 主流程：对每个数据集运行特征选择
# ============================================================
results = {}

for label, fname in FILES.items():
    path = os.path.join(INPUT_DIR, fname)
    print(f"\n{'='*70}")
    print(f"读取: {path}")
    df = pd.read_csv(path)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df['stock_code'] = df['stock_code'].astype(str)
    print(f"shape: {df.shape} | 股票数: {df['stock_code'].nunique()}")

    selected, importance_df, corr_pairs, metadata = feature_selection_pipeline(
        df=df,
        dataset_name=label,
        feature_cols=candidate_features,
        top_k=50,
        verbose=True
    )

    # 加上时间离散特征
    # time_keep = ['year', 'month']
    # all_features = selected + time_keep

    time_keep = [f for f in time_features_numeric if f not in selected]
    all_features = selected + time_keep

    results[label] = {
        'df':           df,
        'selected':     selected,
        'all_features': all_features,
        'importance':   importance_df,
        'corr_pairs':   corr_pairs,
        'metadata':     metadata,
    }

    # 保存重要性
    importance_df.to_csv(
        os.path.join(OUTPUT_DIR, f"importance_{label}.csv"), index=False)

    # 保存相关性对
    if corr_pairs:
        pd.DataFrame(corr_pairs).to_csv(
            os.path.join(OUTPUT_DIR, f"high_corr_pairs_{label}.csv"), index=False)

    # 保存选定特征列表
    feature_info = []
    for i, feat in enumerate(all_features, 1):
        if feat in original_features:       cat = 'Original'
        elif feat in time_features_numeric: cat = 'Temporal'
        elif feat in time_features_cyclic:  cat = 'Temporal_Cyclic'
        elif feat in technical_features:    cat = 'Technical'
        else:                               cat = 'Unknown'
        feature_info.append({'Index': i, 'Feature': feat, 'Category': cat})

    pd.DataFrame(feature_info).to_csv(
        os.path.join(OUTPUT_DIR, f"selected_features_{label}.csv"), index=False)

    print(f"\n  最终特征数: {len(all_features)} (选定{len(selected)} + 时间{len(time_keep)})")

# ============================================================
# 保存汇总配置
# ============================================================
feature_config = {}
for label, res in results.items():
    feature_config[label] = {
        'all_features':      res['all_features'],
        'selected_features': res['selected'],
        # 'temporal_features': ['year', 'month'],
        'temporal_features': time_features_numeric + time_features_cyclic,
        'n_total':           len(res['all_features']),
    }
feature_config['train_end_date'] = TRAIN_END_DATE

with open(os.path.join(OUTPUT_DIR, 'feature_config.json'), 'w', encoding='utf-8') as f:
    json.dump(feature_config, f, indent=2, ensure_ascii=False, cls=NumpyEncoder)
print(f"\n✓ feature_config.json 已保存")

# ============================================================
# 可视化：特征重要性对比
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Feature Importance Analysis - US Stocks', fontsize=14, fontweight='bold')

for idx, (label, res) in enumerate(results.items()):
    ax = axes[idx]
    imp = res['importance'].head(20).copy()
    imp = imp.sort_values('Ensemble_Score', ascending=True)
    colors = ['#2E86AB' if f in res['selected'] else '#CCCCCC' for f in imp['Feature']]
    ax.barh(range(len(imp)), imp['Ensemble_Score'], color=colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels(imp['Feature'], fontsize=9)
    ax.set_xlabel('Ensemble Importance Score')
    ax.set_title(f'({chr(97+idx)}) {label}', fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.set_xlim(0, 1.0)
    legend_elements = [
        Patch(facecolor='#2E86AB', edgecolor='black', label='Selected'),
        Patch(facecolor='#CCCCCC', edgecolor='black', label='Not Selected')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
fig_path = "./visualizations/feature_importance_us.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"✓ 可视化已保存: {fig_path}")
plt.close()

# ============================================================
# 最终汇总
# ============================================================
print("\n" + "=" * 70)
print("✅ 特征选择完成")
print("=" * 70)
for label, res in results.items():
    print(f"\n  {label}:")
    print(f"    最终特征数: {len(res['all_features'])}")
    print(f"    特征列表:   {res['all_features']}")

print(f"\n💾 输出文件:")
print(f"  {OUTPUT_DIR}/importance_*.csv")
print(f"  {OUTPUT_DIR}/selected_features_*.csv")
print(f"  {OUTPUT_DIR}/feature_config.json")
print(f"  ./visualizations/feature_importance_us.png")


美股特征选择（对齐A股逻辑，无石油特征）

特征构成:
  原始市场特征:          8
  时间特征（周期性编码）: 6
  技术指标特征:          29
  候选特征总数:          50

读取: ./stock_data_tushare/us_medical_features_full.csv
shape: (78661, 55) | 股票数: 20

特征选择: us_medical

[步骤0] 目标泄露检查: ✅ 通过

[步骤1] 训练集: 64,261行 | 2010-02-04 ~ 2022-12-30 | 20只股票

[步骤2] 特征矩阵: (64261, 50) | 目标: (64261,)

[步骤3] 缺失值: 0 → 0

[步骤4] 计算特征重要性...
  [1/6] LightGBM...
  [2/6] XGBoost...
  [3/6] Random Forest...
  [4/6] F-statistic...
  [5/6] Mutual Information...
  [6/6] Pearson Correlation...

[步骤5] 高度相关特征对 (|r|>0.9): 79

[步骤7] 筛选结果:
  候选特征: 50
  Top-K初选: 50
  冗余移除: 0
  最终选定: 50

  Top-20特征:
  排名    状态         特征名                       得分        
  ----------------------------------------------------
     1  ✓ 保留       roc_5                     0.5047
     2  ✓ 保留       low                       0.4622
     3  ✓ 保留       close_to_ma20             0.4225
     4  ✓ 保留       roc_20                    0.4161
     5  ✓ 保留       high_low_range            0.3967
     6  ✓ 保留     

In [9]:
"""
美股数据集切分与标准化
对齐A股逻辑（Step 7-8），X/Y物理隔离

输入：
  stock_data_tushare/us_medical_features_full.csv
  stock_data_tushare/us_power_features_full.csv
  stock_data_tushare/feature_selection/feature_config.json

输出：
  stock_data_tushare/processed/us_medical/train_X.csv 等
  stock_data_tushare/processed/us_power/train_X.csv 等

安装依赖：pip install pandas numpy scikit-learn
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import json
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

print("\n" + "=" * 80)
print("美股数据集切分与标准化（X/Y物理隔离）")
print("=" * 80)

# ============================================================
# 配置
# ============================================================
INPUT_DIR    = "./stock_data_tushare"
CONFIG_PATH  = "./stock_data_tushare/feature_selection/feature_config.json"
OUTPUT_BASE  = "./stock_data_tushare/processed"
SCALER_DIR   = "./stock_data_tushare/scalers"

os.makedirs(OUTPUT_BASE, exist_ok=True)
os.makedirs(SCALER_DIR,  exist_ok=True)

# 时间划分（与A股完全一致）
TRAIN_END  = pd.to_datetime('2022-12-31')
VAL_START  = pd.to_datetime('2023-01-01')
VAL_END    = pd.to_datetime('2024-06-30')
TEST_START = pd.to_datetime('2024-07-01')
TEST_END   = pd.to_datetime('2025-11-14')

TARGET_COL = 'pct_chg_next'   # 美股目标列名

# 数据集配置
DATASETS = {
    "us_medical": "us_medical_features_full.csv",
    "us_power":   "us_power_features_full.csv",
}

# ============================================================
# 读取特征配置（来自Step 6特征选择结果）
# ============================================================
print(f"\n[1] 读取特征配置: {CONFIG_PATH}")
if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"找不到特征配置文件，请先运行 feature_selection_us.py")

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    feature_config = json.load(f)

print(f"  配置中的数据集: {[k for k in feature_config.keys() if k != 'train_end_date']}")

# ============================================================
# 核心处理函数
# ============================================================
def process_and_save(df, label, config_key, output_dir):
    print(f"\n{'='*70}")
    print(f"处理: {label}")
    print(f"{'='*70}")

    # ------ 获取特征列表 ------
    cfg = feature_config[config_key]
    selected  = cfg['selected_features']
    temporal  = cfg['temporal_features']          # ['year', 'month']
    feature_cols = cfg['all_features']            # selected + temporal

    print(f"  特征总数: {len(feature_cols)}")
    print(f"  其中时间特征: {temporal}")
    print(f"  目标列: {TARGET_COL}")

    # ------ 检查特征是否存在 ------
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        print(f"  ⚠️  以下特征不在数据中，已跳过: {missing}")
        feature_cols = [c for c in feature_cols if c in df.columns]
        selected     = [c for c in selected     if c in df.columns]

    # 检查目标列
    if TARGET_COL not in df.columns:
        raise ValueError(f"数据中缺少目标列 {TARGET_COL}，请检查特征工程输出")

    # ------ 构造X和y ------
    id_cols = ['stock_code', 'trade_date']
    X_df    = df[id_cols + feature_cols].copy()
    y_df    = df[id_cols + [TARGET_COL]].copy()

    X_df['trade_date'] = pd.to_datetime(X_df['trade_date'])
    y_df['trade_date'] = pd.to_datetime(y_df['trade_date'])

    # ------ 时间切分 ------
    def split_by_time(data):
        train = data[data['trade_date'] <= TRAIN_END].copy()
        val   = data[(data['trade_date'] >= VAL_START) & (data['trade_date'] <= VAL_END)].copy()
        test  = data[(data['trade_date'] >= TEST_START) & (data['trade_date'] <= TEST_END)].copy()
        return train, val, test

    X_train, X_val, X_test = split_by_time(X_df)
    y_train, y_val, y_test = split_by_time(y_df)

    print(f"\n  切分结果:")
    print(f"    Train: {len(X_train):>6,}行  "
          f"({X_train['trade_date'].min().date()} ~ {X_train['trade_date'].max().date()})")
    print(f"    Val:   {len(X_val):>6,}行  "
          f"({X_val['trade_date'].min().date()} ~ {X_val['trade_date'].max().date()})")
    print(f"    Test:  {len(X_test):>6,}行  "
          f"({X_test['trade_date'].min().date()} ~ {X_test['trade_date'].max().date()})")

    # ------ 标准化（时间特征不参与缩放）------
    cols_to_scale = [c for c in feature_cols if c not in temporal]
    print(f"\n  标准化 {len(cols_to_scale)} 个特征（跳过时间特征）...")

    # 缺失值处理（按股票分组前向填充）
    def safe_fill(d):
        d[cols_to_scale] = (
            d.groupby('stock_code')[cols_to_scale]
             .transform(lambda x: x.ffill().fillna(0))
        )
        return d

    X_train = safe_fill(X_train)
    X_val   = safe_fill(X_val)
    X_test  = safe_fill(X_test)

    # 只用训练集fit scaler
    scaler = RobustScaler()
    scaler.fit(X_train[cols_to_scale])

    X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
    X_val[cols_to_scale]   = scaler.transform(X_val[cols_to_scale])
    X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

    print(f"  ✅ 标准化完成（仅用训练集fit，无未来信息泄露）")

    # ------ 保存文件 ------
    os.makedirs(output_dir, exist_ok=True)

    X_train.to_csv(f'{output_dir}/train_X.csv', index=False)
    X_val.to_csv(  f'{output_dir}/val_X.csv',   index=False)
    X_test.to_csv( f'{output_dir}/test_X.csv',  index=False)

    y_train.to_csv(f'{output_dir}/train_y.csv', index=False)
    y_val.to_csv(  f'{output_dir}/val_y.csv',   index=False)
    y_test.to_csv( f'{output_dir}/test_y.csv',  index=False)

    # 保存scaler
    scaler_path = os.path.join(SCALER_DIR, f'scaler_{label}.pkl')
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)

    print(f"\n  💾 已保存至: {output_dir}")
    print(f"     train_X.csv ({len(X_train):,}行) / train_y.csv")
    print(f"     val_X.csv   ({len(X_val):,}行) / val_y.csv")
    print(f"     test_X.csv  ({len(X_test):,}行) / test_y.csv")
    print(f"     scaler: {scaler_path}")

    # ------ 验证：检查泄露 ------
    print(f"\n  验证（防泄露检查）:")
    print(f"    X_train中是否含目标列: {TARGET_COL in X_train.columns}")
    print(f"    y_train中是否含特征:   {any(c in y_train.columns for c in cols_to_scale)}")
    print(f"    Train/Val日期重叠:     "
          f"{X_train['trade_date'].max() >= X_val['trade_date'].min()}")

    return {
        'n_train': len(X_train),
        'n_val':   len(X_val),
        'n_test':  len(X_test),
        'n_features': len(feature_cols),
        'feature_cols': feature_cols,
    }

# ============================================================
# 主流程
# ============================================================
print(f"\n[2] 开始处理数据集...")

summary = {}
for label, fname in DATASETS.items():
    path = os.path.join(INPUT_DIR, fname)
    df = pd.read_csv(path)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df['stock_code'] = df['stock_code'].astype(str)

    output_dir = os.path.join(OUTPUT_BASE, label)
    result = process_and_save(
        df=df,
        label=label,
        config_key=label,
        output_dir=output_dir,
    )
    summary[label] = result

# ============================================================
# 汇总
# ============================================================
print("\n" + "=" * 80)
print("✅ 全部完成")
print("=" * 80)
for label, res in summary.items():
    print(f"\n  {label}:")
    print(f"    特征数:  {res['n_features']}")
    print(f"    Train:   {res['n_train']:,}行")
    print(f"    Val:     {res['n_val']:,}行")
    print(f"    Test:    {res['n_test']:,}行")

print(f"\n  输出目录结构:")
print(f"  stock_data_tushare/processed/")
for label in DATASETS:
    print(f"    {label}/")
    print(f"      train_X.csv / train_y.csv")
    print(f"      val_X.csv   / val_y.csv")
    print(f"      test_X.csv  / test_y.csv")


美股数据集切分与标准化（X/Y物理隔离）

[1] 读取特征配置: ./stock_data_tushare/feature_selection/feature_config.json
  配置中的数据集: ['us_medical', 'us_power']

[2] 开始处理数据集...

处理: us_medical
  特征总数: 50
  其中时间特征: ['year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear', 'weekofyear', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos']
  目标列: pct_chg_next

  切分结果:
    Train: 64,261行  (2010-02-04 ~ 2022-12-30)
    Val:    7,480行  (2023-01-03 ~ 2024-06-28)
    Test:   6,920行  (2024-07-01 ~ 2025-11-13)

  标准化 37 个特征（跳过时间特征）...
  ✅ 标准化完成（仅用训练集fit，无未来信息泄露）

  💾 已保存至: ./stock_data_tushare/processed/us_medical
     train_X.csv (64,261行) / train_y.csv
     val_X.csv   (7,480行) / val_y.csv
     test_X.csv  (6,920行) / test_y.csv
     scaler: ./stock_data_tushare/scalers/scaler_us_medical.pkl

  验证（防泄露检查）:
    X_train中是否含目标列: False
    y_train中是否含特征:   False
    Train/Val日期重叠:     False

处理: us_power
  特征总数: 50
  其中时间特征: ['year', 'month', 'day', 'dayofweek', 'quarter', 'dayofyear

In [10]:
"""
整理脚本：把美股数据复制到datasets目录，对齐A股结构
- 原文件不动
- y文件里pct_chg_next改成pct_chg
- 目录和key改成dataset4_m和dataset5_p
- 生成phase5_data_us.pkl
"""

import pandas as pd
import pickle
import os
import shutil

SRC_BASE   = "./stock_data_tushare/processed"
DST_BASE   = "./datasets/processed"
SCALER_SRC = "./stock_data_tushare/scalers"
SCALER_DST = "./datasets/scalers"
PKL_DST    = "./datasets/phase5_data_us.pkl"

DATASETS = {
    "us_medical": "dataset4_m",
    "us_power":   "dataset5_p",
}

SPLITS = ["train", "val", "test"]

os.makedirs(SCALER_DST, exist_ok=True)

phase_data = {}

for src_key, dst_key in DATASETS.items():
    src_dir = os.path.join(SRC_BASE, src_key)
    dst_dir = os.path.join(DST_BASE, dst_key)
    os.makedirs(dst_dir, exist_ok=True)

    print(f"\n处理: {src_key} → {dst_key}")

    for split in SPLITS:
        # 复制X文件
        x_src = os.path.join(src_dir, f"{split}_X.csv")
        x_dst = os.path.join(dst_dir, f"{split}_X.csv")
        shutil.copy2(x_src, x_dst)
        print(f"  ✓ {split}_X.csv 复制完成")

        # 复制y文件，改列名
        y_src = os.path.join(src_dir, f"{split}_y.csv")
        y_dst = os.path.join(dst_dir, f"{split}_y.csv")
        y_df  = pd.read_csv(y_src)
        y_df  = y_df.rename(columns={"pct_chg_next": "pct_chg"})
        y_df.to_csv(y_dst, index=False)
        print(f"  ✓ {split}_y.csv 复制完成（pct_chg_next→pct_chg）")

    # 复制scaler
    scaler_src = os.path.join(SCALER_SRC, f"scaler_{src_key}.pkl")
    scaler_dst = os.path.join(SCALER_DST, f"scaler_{dst_key}.pkl")
    shutil.copy2(scaler_src, scaler_dst)
    print(f"  ✓ scaler_{dst_key}.pkl 复制完成")

    # 读入数据用于生成pkl
    X_train = pd.read_csv(os.path.join(dst_dir, "train_X.csv"))
    y_train = pd.read_csv(os.path.join(dst_dir, "train_y.csv"))
    X_val   = pd.read_csv(os.path.join(dst_dir, "val_X.csv"))
    y_val   = pd.read_csv(os.path.join(dst_dir, "val_y.csv"))
    X_test  = pd.read_csv(os.path.join(dst_dir, "test_X.csv"))
    y_test  = pd.read_csv(os.path.join(dst_dir, "test_y.csv"))

    phase_data[dst_key] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_val':   X_val,
        'y_val':   y_val,
        'X_test':  X_test,
        'y_test':  y_test,
    }

# 保存pkl
with open(PKL_DST, 'wb') as f:
    pickle.dump(phase_data, f)
print(f"\n✓ 已保存: {PKL_DST}")

print("\n" + "="*60)
print("✅ 全部完成")
print("="*60)
print(f"\n生成的目录结构:")
for src_key, dst_key in DATASETS.items():
    print(f"  datasets/processed/{dst_key}/")
    for split in SPLITS:
        print(f"    {split}_X.csv / {split}_y.csv")
print(f"  datasets/scalers/scaler_dataset4_m.pkl")
print(f"  datasets/scalers/scaler_dataset5_p.pkl")
print(f"  datasets/phase5_data_us.pkl")


处理: us_medical → dataset4_m
  ✓ train_X.csv 复制完成
  ✓ train_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ val_X.csv 复制完成
  ✓ val_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ test_X.csv 复制完成
  ✓ test_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ scaler_dataset4_m.pkl 复制完成

处理: us_power → dataset5_p
  ✓ train_X.csv 复制完成
  ✓ train_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ val_X.csv 复制完成
  ✓ val_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ test_X.csv 复制完成
  ✓ test_y.csv 复制完成（pct_chg_next→pct_chg）
  ✓ scaler_dataset5_p.pkl 复制完成

✓ 已保存: ./datasets/phase5_data_us.pkl

✅ 全部完成

生成的目录结构:
  datasets/processed/dataset4_m/
    train_X.csv / train_y.csv
    val_X.csv / val_y.csv
    test_X.csv / test_y.csv
  datasets/processed/dataset5_p/
    train_X.csv / train_y.csv
    val_X.csv / val_y.csv
    test_X.csv / test_y.csv
  datasets/scalers/scaler_dataset4_m.pkl
  datasets/scalers/scaler_dataset5_p.pkl
  datasets/phase5_data_us.pkl


In [ ]:
---------------------------暂时不用美国的指数-------------------------------

In [9]:
"""
下载美股大盘指数数据
SPY = 标普500 ETF（对应沪深300，大盘）
IWM = 罗素2000 ETF（对应中证1000，小盘）

安装依赖：pip install tushare pandas
"""

import tushare as ts
import pandas as pd
import os
import time

# ============================================================
# ！！！填入你的Token！！！
# ============================================================
TUSHARE_TOKEN = "YOUR_TUSHARE_TOKEN"

START_DATE = "20100101"
END_DATE   = "20241114"
OUTPUT_DIR = "./stock_data_tushare"

# ============================================================
# 主流程
# ============================================================

if __name__ == "__main__":
    ts.set_token(TUSHARE_TOKEN)
    pro = ts.pro_api()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    indices = {
        "SPY": "标普500ETF（大盘）",
        "IWM": "罗素2000ETF（小盘）",
    }

    results = {}

    for ticker, name in indices.items():
        print(f"下载 {ticker} {name} ...")
        try:
            df = pro.us_daily(
                ts_code=ticker,
                start_date=START_DATE,
                end_date=END_DATE,
                fields="trade_date,open,high,low,close,vol"
            )

            if df is None or df.empty:
                print(f"  [空数据] {ticker}")
                continue

            df = df.rename(columns={"trade_date": "date", "vol": "volume"})
            df["date"] = pd.to_datetime(df["date"])
            df = df.set_index("date").sort_index()

            # 计算涨跌幅（后续合并到个股数据用）
            df["pct_chg"] = df["close"].pct_change() * 100

            # 保存
            save_path = os.path.join(OUTPUT_DIR, f"{ticker}.csv")
            df.to_csv(save_path)
            results[ticker] = df

            print(f"  [OK] {ticker}: {len(df)}个交易日，"
                  f"{df.index.min().date()} ~ {df.index.max().date()}")

        except Exception as e:
            print(f"  [失败] {ticker}: {e}")

        time.sleep(0.5)

    # 合并成一个文件，只保留涨跌幅列，方便后续merge
    if len(results) == 2:
        spy_pct = results["SPY"]["pct_chg"].rename("spy_pct_chg")
        iwm_pct = results["IWM"]["pct_chg"].rename("iwm_pct_chg")
        combined = pd.concat([spy_pct, iwm_pct], axis=1)
        combined_path = os.path.join(OUTPUT_DIR, "us_market_index.csv")
        combined.to_csv(combined_path)
        print(f"\n合并指数文件已保存: {combined_path}")
        print(combined.tail())
    elif len(results) == 1:
        ticker = list(results.keys())[0]
        print(f"\n只成功下载了{ticker}，另一个需要检查")

    print("\n完成！")

下载 SPY 标普500ETF（大盘） ...
  [OK] SPY: 2737个交易日，2014-01-02 ~ 2024-11-14
下载 IWM 罗素2000ETF（小盘） ...
  [OK] IWM: 2737个交易日，2014-01-02 ~ 2024-11-14

合并指数文件已保存: ./stock_data_tushare\us_market_index.csv
            spy_pct_chg  iwm_pct_chg
date                                
2024-11-08     0.433169     0.736103
2024-11-11     0.095287     1.503444
2024-11-12    -0.310642    -1.754241
2024-11-13     0.048584    -0.989640
2024-11-14    -0.643011    -1.348305

完成！
